In [1]:
from datetime import datetime
import pandas as pd
import glob

now = datetime.now().strftime("%Y-%m-%d_%H:%M:%s")

print(f"Notebook run: {now}")

Notebook run: 2026-03-09_20:23:1773084210


In [2]:
# Specify your folder path here
folder_path = "/mnt/data/StressID-img-mtcnn-data"  # ← CHANGE THIS to your actual folder path

# Get all PNG files recursively from the specified folder
png_files = glob.glob(f"{folder_path}/**/*.png", recursive=True)

# Create DataFrame with column named 'path'
df = pd.DataFrame({"path": png_files})

print(f"Found {len(df)} PNG files")
print(df.head())   

Found 271158 PNG files
                                                path
0  /mnt/data/StressID-img-mtcnn-data/no-stress/g9...
1  /mnt/data/StressID-img-mtcnn-data/no-stress/g9...
2  /mnt/data/StressID-img-mtcnn-data/no-stress/g9...
3  /mnt/data/StressID-img-mtcnn-data/no-stress/g9...
4  /mnt/data/StressID-img-mtcnn-data/no-stress/g9...


In [3]:
df['str-label'] = df['path'].str.split('/').str[-4]
label_map = {'stress': 1.0, 'no-stress': 0.0}
df['label'] = df['str-label'].map(label_map)
# Extract just the filename (with extension)
df['subject'] = df['path'].str.split('/').str[-3]

print(f"df shape: {df.shape}")
print(f"num of subjects: {df['subject'].nunique()}")
print(f"training-subjects: {df['subject'].nunique()*.9}")
print(f"test-subjects: {df['subject'].nunique()*.1}")
df.sample(3)

df shape: (271158, 4)
num of subjects: 53
training-subjects: 47.7
test-subjects: 5.300000000000001


,path,str-label,label,subject
67311,/mnt/data/StressID-img-mtcnn-data/no-stress/8g...,no-stress,0.0,8g4y
35916,/mnt/data/StressID-img-mtcnn-data/no-stress/r5...,no-stress,0.0,r5s8
221469,/mnt/data/StressID-img-mtcnn-data/stress/x1q3/...,stress,1.0,x1q3


In [4]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# First split: 80% train, 20% temp
splitter1 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, temp_idx = next(splitter1.split(df, groups=df['subject']))

# Second split: split the 20% into two 10% parts
splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
_, temp_split = next(splitter2.split(df.iloc[temp_idx], groups=df.iloc[temp_idx]['subject']))

# Map back to original indices
val_idx = temp_idx[temp_split]
test_idx = np.setdiff1d(temp_idx, val_idx)

# Final datasets
train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]
test_df = df.iloc[test_idx]   
print(f"Train subjects: {train_df['subject'].nunique()}")
print(f"Val subjects: {val_df['subject'].nunique()}")   
print(f"Test subjects: {test_df['subject'].nunique()}")   

Train subjects: 39
Val subjects: 7
Test subjects: 7


In [5]:
# Check subject distribution
print("Training subjects:", train_df['subject'].nunique())
print("Validation subjects:", val_df['subject'].nunique())
print("Test subjects:", test_df['subject'].nunique())  # Added

# Verify no subject overlap
common_train_val = set(train_df['subject']) & set(val_df['subject'])
common_train_test = set(train_df['subject']) & set(test_df['subject'])  
common_val_test = set(val_df['subject']) & set(test_df['subject'])
print("Train-Val shared subjects:", common_train_val)  # Should be empty
print("Train-Test shared subjects:", common_train_test)  # Should be empty  
print("Val-Test shared subjects:", common_val_test)  # Should be empty

# Check label distribution
print("\nTrain label balance:")
print(train_df['label'].value_counts(normalize=True))
print("\nVal label balance:")
print(val_df['label'].value_counts(normalize=True))
print("\nTest label balance:")  # Added
print(test_df['label'].value_counts(normalize=True))   

Training subjects: 39
Validation subjects: 7
Test subjects: 7
Train-Val shared subjects: set()
Train-Test shared subjects: set()
Val-Test shared subjects: set()

Train label balance:
label
0.0    0.556017
1.0    0.443983
Name: proportion, dtype: float64

Val label balance:
label
0.0    0.631165
1.0    0.368835
Name: proportion, dtype: float64

Test label balance:
label
0.0    0.591419
1.0    0.408581
Name: proportion, dtype: float64


In [6]:
# Check for any overlapping subjects between splits
train_subjects = set(train_df['subject'])
val_subjects = set(val_df['subject'])
test_subjects = set(test_df['subject'])

# Verify no subject appears in multiple splits
assert train_subjects.isdisjoint(val_subjects), "Train and Val have overlapping subjects"
assert train_subjects.isdisjoint(test_subjects), "Train and Test have overlapping subjects"  
assert val_subjects.isdisjoint(test_subjects), "Val and Test have overlapping subjects"

print("No subject leakage detected - all splits are clean!")   

No subject leakage detected - all splits are clean!


In [7]:
import os 

# Save each split to CSV
train_df.to_csv(os.path.join(folder_path, 'train_split.csv'), sep=";", index=False)
val_df.to_csv(os.path.join(folder_path, 'val_split.csv'), sep=";", index=False) 
test_df.to_csv(os.path.join(folder_path, 'test_split.csv'), sep=";", index=False)

print(" Splits saved as CSV files!")   

 Splits saved as CSV files!
